In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch_numopt
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import *
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from train_loop import train_loop

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
class Net(nn.Module):
    def __init__(self, input_size, device="cpu"):
        super().__init__()
        self.f1 = nn.Linear(input_size, 10, device=device)
        self.f2 = nn.Linear(10, 20, device=device)
        self.f3 = nn.Linear(20, 20, device=device)
        self.f4 = nn.Linear(20, 10, device=device)
        self.f5 = nn.Linear(10, 1, device=device)

        self.activation = nn.ReLU()
        # self.activation = nn.Sigmoid()

    def forward(self, x):
        x = self.activation(self.f1(x))
        x = self.activation(self.f2(x))
        x = self.activation(self.f3(x))
        x = self.activation(self.f4(x))
        x = self.f5(x)

        return x

In [4]:
# X, y = load_diabetes(return_X_y=True, scaled=False)
# X, y = make_regression(n_samples=1000, n_features=100)
X, y = make_friedman1(n_samples=1000, noise=1e-2)

X_scaler = MinMaxScaler()
X = X_scaler.fit_transform(X)
X = torch.Tensor(X).to(device)

y_scaler = MinMaxScaler()
y = y_scaler.fit_transform(y.reshape((-1, 1)))
y = torch.Tensor(y).to(device)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

torch_data = TensorDataset(X_train, y_train)
data_loader = DataLoader(torch_data, batch_size=1000)

torch.Size([800, 10]) torch.Size([800, 1])
torch.Size([200, 10]) torch.Size([200, 1])


In [5]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.ConjugateGradient(
    model=model,
    lr=10,
    lr_init="lipschiz",
    c1=1e-4,
    tau=0.1,
    line_search_method="backtrack",
    line_search_cond="armijo",
    cg_method="PR"
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.1929231435060501
epoch:  1, loss: 0.11830464750528336
epoch:  2, loss: 0.07753029465675354
epoch:  3, loss: 0.05601980537176132
epoch:  4, loss: 0.04505040869116783
epoch:  5, loss: 0.03961930051445961
epoch:  6, loss: 0.03699234873056412
epoch:  7, loss: 0.03574525564908981
epoch:  8, loss: 0.03516281396150589
epoch:  9, loss: 0.03489236906170845
epoch:  10, loss: 0.034767091274261475
epoch:  11, loss: 0.034708280116319656
epoch:  12, loss: 0.03467980772256851
epoch:  13, loss: 0.03466527536511421
epoch:  14, loss: 0.03465697169303894
epoch:  15, loss: 0.034641899168491364
epoch:  16, loss: 0.0346258208155632
epoch:  17, loss: 0.0346246100962162
epoch:  18, loss: 0.03458726406097412
epoch:  19, loss: 0.034584179520606995
epoch:  20, loss: 0.03450814634561539
epoch:  21, loss: 0.03447389230132103
epoch:  22, loss: 0.034457042813301086
epoch:  23, loss: 0.03444809466600418
epoch:  24, loss: 0.034435924142599106
epoch:  25, loss: 0.0344189815223217
epoch:  26, loss: 0.

In [6]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.745123028755188
Test metrics:  R2 = 0.7208229303359985


In [7]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.ConjugateGradient(
    model=model,
    lr=10,
    lr_init="BB",
    c1=1e-4,
    tau=0.1,
    line_search_method="backtrack",
    line_search_cond="armijo",
    cg_method="FR"
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.3782646059989929
epoch:  1, loss: 0.21875730156898499
epoch:  2, loss: 0.13161790370941162
epoch:  3, loss: 0.0844137966632843
epoch:  4, loss: 0.05948954448103905
epoch:  5, loss: 0.046704310923814774
epoch:  6, loss: 0.04031617194414139
epoch:  7, loss: 0.03719399869441986
epoch:  8, loss: 0.035693585872650146
epoch:  9, loss: 0.034980859607458115
epoch:  10, loss: 0.034644514322280884
epoch:  11, loss: 0.03448576480150223
epoch:  12, loss: 0.034410037100315094
epoch:  13, loss: 0.03437285125255585
epoch:  14, loss: 0.034353502094745636
epoch:  15, loss: 0.03434242308139801
epoch:  16, loss: 0.03432605415582657
epoch:  17, loss: 0.03430568426847458
epoch:  18, loss: 0.03429410234093666
epoch:  19, loss: 0.034277066588401794
epoch:  20, loss: 0.034255705773830414
epoch:  21, loss: 0.03424359858036041
epoch:  22, loss: 0.03422672674059868
epoch:  23, loss: 0.03420405834913254
epoch:  24, loss: 0.03419134393334389
epoch:  25, loss: 0.03417440876364708
epoch:  26, loss

In [8]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.7714645862579346
Test metrics:  R2 = 0.7274962067604065
